# ทดสอบ koharu บน Google Colab (T4 รุ่นฟรี) — รันครั้งเดียว

โน้ตบุ๊กนี้ build koharu-port (branch `port-0.83` commit `da18f1d8`) จาก GitHub แล้วรัน pipeline เต็ม
**detect → OCR → แปล (LLM ในเครื่อง Colab) → inpaint → render** กับภาพตัวอย่าง 5–10 หน้า
และวัดเวลาแยกตามขั้น พร้อมบันทึกข้อความญี่ปุ่น→ไทยไว้ให้ตรวจคุณภาพ

**กติกา (ห้ามข้าม)**
- ใช้ภาพจาก **มังงะปกติ** 5–10 หน้าเท่านั้น ห้ามใช้โดจินหรือเนื้อหา 18+ เพราะบัญชี Google ผูกกับ GA4/GSC อยู่
- โน้ตบุ๊กนี้ **ไม่ต่อกับระบบจริง** ไม่มี manga-th.net, API, R2, ฟาร์ม หรือคิวงาน ไม่เปิด server หรือ tunnel ใด ๆ
- ใช้บัญชี Google บัญชีเดียว ถ้าโควตา GPU หมดให้หยุดแล้วรอวันอื่น ห้ามสลับบัญชี
- ไม่ต้องใส่ token หรือรหัสผ่านใด ๆ (repo และโมเดลทั้งหมดเป็น public)

**ก่อนเริ่ม:** เมนู Runtime → Change runtime type → เลือก **T4 GPU** → Save
แล้วกดรันทีละช่องจากบนลงล่าง (Shift+Enter) อ่านคู่มือ `README.md` ประกอบ

## ขั้น 1 — ตรวจ GPU และไดรเวอร์ (ไม่ถึง 1 นาที)
koharu ต้องการไดรเวอร์ที่รองรับ CUDA **13.3** ขึ้นไป ช่องนี้จะบอกว่า Colab ตอนนี้มีเวอร์ชันไหน

In [ ]:
import os, re, subprocess, time, glob, shutil

def sh(cmd, env=None):
    """รันคำสั่ง shell แล้วหยุดทันทีถ้าล้มเหลว"""
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f'คำสั่งล้มเหลว: {cmd}')
    return r.stdout

smi = sh('nvidia-smi')
print(smi)
assert 'T4' in smi, 'ยังไม่ได้เลือก GPU T4: Runtime → Change runtime type → T4 GPU'
m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi)
DRIVER_CUDA = (int(m.group(1)), int(m.group(2)))
print('ไดรเวอร์รองรับ CUDA', DRIVER_CUDA)
print('→ ต้องลง cuda-compat ในขั้น 2' if DRIVER_CUDA < (13, 3) else '→ พอแล้ว ขั้น 2 จะข้ามเอง')

## ขั้น 2 — ทำให้ไดรเวอร์รองรับ CUDA 13.3 (ถ้าจำเป็น, 1–2 นาที)
ลองทางแก้ A ก่อน: ลงแพ็กเกจ `cuda-compat` ของ NVIDIA (forward compatibility ใช้ได้กับการ์ดดาต้าเซ็นเตอร์อย่าง T4)
ถ้าไม่ได้ผลและไดรเวอร์เป็น 13.x อยู่แล้ว จะใช้ทางแก้ B: ลดเกณฑ์เวอร์ชันในสำเนาโค้ดบน Colab (CUDA 13 รุ่นย่อยใช้ข้ามกันได้)
**จดลงแบบฟอร์มว่าใช้ทางแก้ไหน**

In [ ]:
CUDA_FIX = 'ไม่ต้องแก้'
PATCH_MIN_DRIVER = False
PROBE = ('import ctypes; c = ctypes.CDLL("libcuda.so.1"); r = c.cuInit(0); '
         'v = ctypes.c_int(); c.cuDriverGetVersion(ctypes.byref(v)); print(r, v.value)')

def driver_version():
    init, version = sh(f"python3 -c '{PROBE}'", env=os.environ.copy()).split()
    return int(init), int(version)

if DRIVER_CUDA < (13, 3):
    os_id = dict(l.split('=', 1) for l in open('/etc/os-release').read().replace('"', '').split('\n') if '=' in l)
    repo = 'ubuntu' + os_id['VERSION_ID'].replace('.', '')
    sh(f'wget -q https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/cuda-keyring_1.1-1_all.deb '
       '&& dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null && apt-get -qq update')
    found = sh(r"apt-cache search --names-only '^cuda-compat-13-[0-9]+$' | cut -d' ' -f1").split()
    usable = [p for p in found if int(p.rsplit('-', 1)[1]) >= 3]
    if usable:
        pkg = max(usable, key=lambda p: int(p.rsplit('-', 1)[1]))
        sh(f'apt-get -qq install -y {pkg} > /dev/null')
        compat = glob.glob('/usr/local/cuda-13.*/compat')[0]
        os.environ['LD_LIBRARY_PATH'] = compat + ':' + os.environ.get('LD_LIBRARY_PATH', '')
        init, version = driver_version()
        print(f'{pkg}: cuInit={init} driver={version}')
        if init == 0 and version >= 13030:
            CUDA_FIX = f'A: {pkg}'
        else:
            os.environ['LD_LIBRARY_PATH'] = os.environ['LD_LIBRARY_PATH'].split(':', 1)[1]
    if CUDA_FIX == 'ไม่ต้องแก้':
        if DRIVER_CUDA >= (13, 0):
            PATCH_MIN_DRIVER = True
            CUDA_FIX = 'B: ลดเกณฑ์ MIN_DRIVER_VERSION ในสำเนาบน Colab'
        else:
            raise SystemExit('ไดรเวอร์เก่ากว่า CUDA 13 และ cuda-compat ใช้ไม่ได้ → จดลงแบบฟอร์มว่า "ทดสอบไม่ได้: ไดรเวอร์" แล้วหยุดที่นี่')
print('ผล:', CUDA_FIX)

## ขั้น 3 — ลงเครื่องมือ build และฟอนต์ไทย (3–5 นาที)
Rust, clang, Vulkan แบบ CPU (lavapipe) ไว้ให้ตัว render ใช้บนเครื่องที่ไม่มีจอ และฟอนต์ **Noto Sans Thai**

koharu วาดภาษาไทยด้วยฟอนต์ไทยของระบบ (บน Linux หาผ่าน fontconfig) Colab ไม่มีฟอนต์ไทยเลย รอบก่อนจึงออกมาเป็นกล่อง
ฟาร์มบน Windows น่าจะได้ **Leelawadee UI** ของ Windows ซึ่งเอามาใช้บน Colab ไม่ได้ (ลิขสิทธิ์ Microsoft)
จึงใช้ Noto Sans Thai ที่หน้าตาใกล้เคียงที่สุดแทน ท้ายช่องจะวาดตัวอย่างภาษาไทยให้ดูและเตือนถ้ายังเป็นกล่อง

In [ ]:
sh('apt-get -qq update && apt-get -qq install -y libclang-dev clang cmake pkg-config '
   'libfontconfig1-dev libdbus-1-dev libvulkan1 mesa-vulkan-drivers vulkan-tools > /dev/null')
sh('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null')
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
print(sh('rustc --version'))
print(sh('vulkaninfo --summary 2>/dev/null | grep -E "deviceName|driverName" || true'))

# Only one Thai family is installed, so fontconfig's Thai fallback can only pick this one.
THAI_FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/notosansthai/NotoSansThai%5Bwdth,wght%5D.ttf'
sh(f'mkdir -p /usr/share/fonts/truetype/noto-thai && wget -q -O /usr/share/fonts/truetype/noto-thai/NotoSansThai.ttf "{THAI_FONT_URL}" && fc-cache -f')

from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

def check_thai_font():
    """วาดตัวอักษรไทยด้วยฟอนต์ที่ fontconfig เลือกให้ภาษาไทย แล้วเทียบกับรูปกล่อง (glyph ที่ไม่มีในฟอนต์)"""
    families = sh('fc-list :lang=th family').strip()
    path = sh("fc-match -f '%{file}' ':lang=th'").strip()
    print('ฟอนต์ไทยในระบบ:', families or '(ไม่มี)')
    print('ฟอนต์ที่ใช้แทนภาษาไทย:', path)
    font = ImageFont.truetype(path, 48)
    def glyph(ch):
        im = Image.new('L', (72, 72))
        ImageDraw.Draw(im).text((8, 4), ch, font=font, fill=255)
        return im.tobytes()
    box = glyph('')
    ok = bool(families) and glyph('ก') != glyph('ข') and all(glyph(c) != box for c in 'กขคภาษาไทยครับค่ะ')
    sample = Image.new('RGB', (640, 80), 'white')
    ImageDraw.Draw(sample).text((10, 8), 'ทดสอบภาษาไทย สวัสดีครับ ค่ะ 123', font=font, fill='black')
    display(sample)
    print('✅ ฟอนต์ไทยใช้ได้' if ok else '❌ ภาษาไทยยังเป็นกล่อง — อย่าไปต่อ ให้ copy ข้อความในช่องนี้ส่งมา')
    return ok

THAI_FONT_OK = check_thai_font()
assert THAI_FONT_OK, 'ไม่มีฟอนต์ไทย'

## ขั้น 4 — ดึงโค้ด koharu-port (1 นาที)
ตรึงไว้ที่ commit `da18f1d8` เพื่อให้ตรงกับฟาร์มตอนเตรียมการทดสอบ

In [ ]:
COMMIT = 'da18f1d8f911e0166323f66a0a9b5de59e0e3484'
if not os.path.isdir('/content/koharu'):
    sh('git clone -q --filter=blob:none -b port-0.83 https://github.com/blakmak1015/koharu.git /content/koharu')
sh(f'cd /content/koharu && git checkout -q {COMMIT}')
if PATCH_MIN_DRIVER:
    cuda_rs = '/content/koharu/crates/koharu-runtime/src/hardware/cuda.rs'
    src = open(cuda_rs).read()
    assert 'const MIN_DRIVER_VERSION: c_int = 13030;' in src
    open(cuda_rs, 'w').write(src.replace('const MIN_DRIVER_VERSION: c_int = 13030;',
                                         'const MIN_DRIVER_VERSION: c_int = 13000;'))
    print('ใช้ทางแก้ B แล้ว')
print(sh('cd /content/koharu && git log -1 --oneline'))

## ขั้น 5 — เขียนตัววัดผล
โปรแกรมเล็ก ๆ ที่โหลด pipeline ครั้งเดียวแล้วแปลทุกหน้าต่อกัน (หน้าแรก = เย็น/โหลดโมเดล, หน้าถัดไป = อุ่นแล้วแบบฟาร์ม)
แยกเวลาโหลดโมเดลกับเวลาทำงานจริงของแต่ละขั้น เก็บงานไว้ในโฟลเดอร์ผล (`session.khrproj`) เพื่อ render ใหม่ได้โดยไม่ต้องแปลซ้ำ
และจะหยุดทันทีถ้า koharu มองไม่เห็นฟอนต์ไทย ไม่ต้องแก้อะไรในช่องนี้

ไฟล์ที่ได้ต่อรอบ (ในโฟลเดอร์ `out_A` / `out_B`):
- `<หน้า>.png` ภาพที่แปลแล้ว และ `clean_<หน้า>.png` ภาพที่ลบตัวอักษรแล้ว (ยังไม่วางคำแปล)
- `blocks.json` ตำแหน่งคำแปลต่อบล็อก: `page`, `block`, `x` `y` `w` `h` (สัดส่วน 0–1 ของภาพ), `writing_mode`,
  `font_size_px` / `font_size_rel` (ขนาดที่ renderer เลือก, rel = ส่วนของความสูงภาพ), `angle_degrees`, `thai`, `japanese`
  อ่านจากผลที่ renderer จัดวางแล้วแบบเดียวกับที่ฟาร์มส่งขึ้นเว็บ เอาไปวางทับ `clean_*.png` ได้ตรงตำแหน่ง
- `timings.tsv`, `texts.tsv`, `vram.log`, `log.txt`

In [ ]:
%%writefile /content/koharu/crates/koharu-pipeline/src/bin/colab_batch.rs
//! Colab benchmark harness: copied into `crates/koharu-pipeline/src/bin/` by
//! `koharu_colab_benchmark.ipynb` and never built in the farm checkout.
//!
//! Differs from `run.rs` on purpose: one process and one `Pipeline` for every
//! page, so pages after the first measure warm throughput the way the farm
//! runs, and the pipeline comes from `~/.koharu/config.toml` (the notebook
//! writes the farm's settings there) instead of CLI flags. The session is kept
//! on disk so `--render-only` can redraw the pages without loading any model.

use std::{
    collections::BTreeMap,
    fmt::Write as _,
    fs,
    path::{Path, PathBuf},
    sync::{Arc, Mutex},
    time::Instant,
};

use anyhow::{Context as _, Result};
use clap::Parser;
use koharu_pipeline::{Committer, Operation, Pipeline, Progress, Request, Scope, StageOutput};
use koharu_rasterizer::{RasterOptions, Rasterizer};
use koharu_renderer::{FontSource, Frame, LayerKind, Renderer, WritingMode};
use koharu_scene::{AssetInput, AssetMetadata, AssetRole, At, EntityId, PageDraft, Session, Snapshot};

const SESSION: &str = "session.khrproj";

#[derive(Debug, Parser)]
struct Arguments {
    /// Folder of source pages; processed in file-name order.
    #[arg(short, long, required_unless_present = "render_only")]
    input: Option<PathBuf>,
    /// Receives one PNG per page, `timings.tsv`, `texts.tsv` and the session.
    #[arg(short, long)]
    output: PathBuf,
    /// Redraw every page of the saved session in `--output`; no model is loaded.
    #[arg(long)]
    render_only: bool,
}

struct SessionCommitter<'a>(&'a mut Session);

#[async_trait::async_trait]
impl Committer for SessionCommitter<'_> {
    async fn commit(&mut self, output: StageOutput) -> Result<koharu_scene::Snapshot> {
        Ok(self.0.commit(output.patch).await?.snapshot)
    }
}

#[tokio::main]
async fn main() -> Result<()> {
    let arguments = Arguments::parse();
    let renderer = Renderer::new()?;
    let rasterizer = Rasterizer::new()?;
    require_thai_font(&renderer).await?;

    let mut blocks = Vec::new();
    if arguments.render_only {
        let session = Session::open(arguments.output.join(SESSION)).await?;
        let snapshot = session.snapshot();
        for page in snapshot.pages() {
            let label = page.page()?.label;
            let seconds =
                export_page(&renderer, &rasterizer, &snapshot, page.id(), &label, &arguments.output, &mut blocks)
                    .await?;
            eprintln!("{label}: render {seconds:.2}s");
        }
        write_blocks(&arguments.output, &blocks)?;
        return Ok(());
    }

    let input = arguments.input.as_ref().expect("clap requires --input");
    fs::create_dir_all(&arguments.output)?;
    let mut inputs = fs::read_dir(input)?
        .filter_map(|entry| entry.ok().map(|entry| entry.path()))
        .filter(|path| media_type(path).is_some())
        .collect::<Vec<_>>();
    inputs.sort();
    anyhow::ensure!(!inputs.is_empty(), "no images in {}", input.display());

    koharu_ml::init().await?;
    let pipeline = Pipeline::load(koharu_ml::device(false))?;
    let mut session = Session::create(arguments.output.join(SESSION)).await?;

    let mut timings = String::from("page\tstage\tmodel\tload_s\trun_s\n");
    let mut texts = String::from("page\tblock\tjapanese\tthai\n");
    for (number, path) in inputs.iter().enumerate() {
        let name = path.file_name().and_then(|n| n.to_str()).unwrap_or("page");
        let page = add_page(&mut session, path, name).await?;

        let events = Arc::new(Mutex::new(Vec::<(Instant, Progress)>::new()));
        let sink = events.clone();
        let started = Instant::now();
        let snapshot = session.snapshot();
        let mut committer = SessionCommitter(&mut session);
        pipeline
            .execute(
                snapshot,
                Request {
                    operation: Operation::Full,
                    scope: Scope::Pages(vec![page]),
                    progress: Some(Arc::new(move |event| {
                        sink.lock().unwrap().push((Instant::now(), event));
                    })),
                    ..Request::default()
                },
                &mut committer,
            )
            .await?;

        // Loading -> Running is model load, Running -> Finished is the work itself.
        let mut loading = BTreeMap::new();
        let mut running = BTreeMap::new();
        for (at, event) in events.lock().unwrap().iter() {
            match event {
                Progress::Loading { stage, .. } => {
                    loading.insert(stage.to_string(), *at);
                }
                Progress::Running { stage, .. } => {
                    running.insert(stage.to_string(), *at);
                }
                Progress::Finished { stage, model, .. } => {
                    let stage = stage.to_string();
                    let (Some(load), Some(run)) = (loading.get(&stage), running.get(&stage)) else {
                        continue;
                    };
                    let load_s = (*run - *load).as_secs_f64();
                    let run_s = (*at - *run).as_secs_f64();
                    writeln!(timings, "{name}\t{stage}\t{model}\t{load_s:.2}\t{run_s:.2}")?;
                }
                _ => {}
            }
        }

        let snapshot = session.snapshot();
        let render_s =
            export_page(&renderer, &rasterizer, &snapshot, page, name, &arguments.output, &mut blocks).await?;
        writeln!(timings, "{name}\trender\tkoharu-renderer\t0.00\t{render_s:.2}")?;
        let total = started.elapsed().as_secs_f64();
        writeln!(timings, "{name}\ttotal\t-\t-\t{total:.2}")?;
        eprintln!("[{}/{}] {name}: {total:.1}s", number + 1, inputs.len());

        if let Some(group) = snapshot.page(page)?.text_group()? {
            for (block, layer) in group.text_layers()?.enumerate() {
                let content = layer.content()?;
                let source = content.source()?.map(|s| s.text.value).unwrap_or_default();
                let thai = content.translation()?.map(|t| t.text.value).unwrap_or_default();
                writeln!(texts, "{name}\t{}\t{}\t{}", block + 1, one_line(&source), one_line(&thai))?;
            }
        }
    }

    fs::write(arguments.output.join("timings.tsv"), timings)?;
    fs::write(arguments.output.join("texts.tsv"), texts)?;
    write_blocks(&arguments.output, &blocks)?;
    Ok(())
}

/// Writes `<stem>.png` (translated), `clean_<stem>.png` (text removed) and
/// appends the page's text blocks; returns the seconds the translated render took.
///
/// Blocks are read the way the farm's `farm_page_blocks` publishes them: from
/// the frame the renderer resolved, not the scene, which still holds the
/// Japanese balloon, its vertical writing mode and no font size.
async fn export_page(
    renderer: &Renderer,
    rasterizer: &Rasterizer,
    snapshot: &Snapshot,
    page: EntityId,
    name: &str,
    output: &Path,
    blocks: &mut Vec<String>,
) -> Result<f64> {
    let stem = stem(Path::new(name));
    let started = Instant::now();
    let frame = renderer.render(snapshot, page).await?;
    save(rasterizer, &frame, &output.join(format!("{stem}.png")))?;
    let seconds = started.elapsed().as_secs_f64();

    let clean = renderer.render_without_text(snapshot, page).await?;
    save(rasterizer, &clean, &output.join(format!("clean_{stem}.png")))?;

    let size = snapshot.page(page)?.page()?;
    let (width, height) = (size.width as f32, size.height as f32);
    let mut block = 0;
    for layer in frame.layers() {
        let LayerKind::Text(text) = layer.kind() else {
            continue;
        };
        let thai = text.text.trim();
        let bounds = text.layout_bounds;
        if !layer.presentation().visible || thai.is_empty() || !(bounds.width > 0.0 && bounds.height > 0.0) {
            continue;
        }
        block += 1;
        let japanese = snapshot
            .text_layer(layer.entity())
            .and_then(|layer| layer.content())
            .and_then(|content| content.source())
            .ok()
            .flatten()
            .map(|source| source.text.value)
            .unwrap_or_default();
        let writing_mode = match text.writing_mode {
            WritingMode::Horizontal => "horizontal",
            WritingMode::VerticalRl => "vertical-rl",
        };
        blocks.push(format!(
            r#"{{"page":{},"block":{block},"x":{:.5},"y":{:.5},"w":{:.5},"h":{:.5},"writing_mode":"{writing_mode}","font_size_px":{:.2},"font_size_rel":{:.5},"angle_degrees":{:.2},"thai":{},"japanese":{}}}"#,
            json_string(name),
            bounds.x / width,
            bounds.y / height,
            bounds.width / width,
            bounds.height / height,
            text.font_size,
            text.font_size / height,
            text.angle_degrees,
            json_string(thai),
            json_string(japanese.trim()),
        ));
    }
    Ok(seconds)
}

fn save(rasterizer: &Rasterizer, frame: &Frame, out: &Path) -> Result<()> {
    let raster = rasterizer.rasterize(&frame.raster_frame()?, RasterOptions::default())?;
    raster.image.save(out).with_context(|| format!("failed to write {}", out.display()))
}

fn write_blocks(output: &Path, blocks: &[String]) -> Result<()> {
    Ok(fs::write(output.join("blocks.json"), format!("[\n{}\n]\n", blocks.join(",\n")))?)
}

/// `koharu-pipeline` has no JSON dependency, and this is the only JSON written.
fn json_string(text: &str) -> String {
    let mut escaped = String::with_capacity(text.len() + 2);
    escaped.push('"');
    for character in text.chars() {
        match character {
            '"' => escaped.push_str("\\\""),
            '\\' => escaped.push_str("\\\\"),
            c if u32::from(c) < 0x20 => escaped.push_str(&format!("\\u{:04x}", u32::from(c))),
            c => escaped.push(c),
        }
    }
    escaped.push('"');
    escaped
}

#[test]
fn json_string_escapes_quotes_backslashes_and_controls() {
    assert_eq!(json_string("ก\"ข\\\n\u{1}"), r#""ก\"ข\\\u000a\u0001""#);
}

/// Thai reaches the renderer only through system script fallback (fontconfig
/// on Linux), so a machine without a Thai system font draws every glyph as a box.
async fn require_thai_font(renderer: &Renderer) -> Result<()> {
    let thai = renderer
        .available_fonts()
        .await?
        .into_iter()
        .filter(|family| family.sources.contains(&FontSource::System) && family.name.contains("Thai"))
        .map(|family| family.name)
        .collect::<Vec<_>>();
    anyhow::ensure!(!thai.is_empty(), "no Thai system font: Thai text would render as boxes");
    eprintln!("Thai system fonts: {}", thai.join(", "));
    Ok(())
}


async fn add_page(session: &mut Session, path: &Path, name: &str) -> Result<EntityId> {
    let source = fs::read(path).with_context(|| format!("failed to read {}", path.display()))?;
    let decoded = image::load_from_memory(&source).context("failed to decode input image")?;
    let mut page = None;
    let patch = session.snapshot().patch(|edit| {
        let id = edit.add_page(
            PageDraft::new(name, f64::from(decoded.width()), f64::from(decoded.height())),
            At::End,
        )?;
        edit.set_asset(
            id,
            &AssetRole::new("source")?,
            AssetInput::new(
                Arc::<[u8]>::from(source),
                media_type(path).expect("inputs are filtered by media type"),
                AssetMetadata {
                    width: Some(decoded.width()),
                    height: Some(decoded.height()),
                    attributes: BTreeMap::new(),
                },
            ),
        )?;
        page = Some(id);
        Ok(())
    })?;
    session.commit(patch).await?;
    Ok(page.expect("page ID is assigned by the edit"))
}

fn media_type(path: &Path) -> Option<&'static str> {
    match path.extension()?.to_str()?.to_ascii_lowercase().as_str() {
        "jpg" | "jpeg" => Some("image/jpeg"),
        "png" => Some("image/png"),
        "webp" => Some("image/webp"),
        _ => None,
    }
}

fn stem(path: &Path) -> &str {
    path.file_stem().and_then(|s| s.to_str()).unwrap_or("page")
}

fn one_line(text: &str) -> String {
    text.replace(['\t', '\n', '\r'], " ")
}

## ขั้น 6 — build (30–60 นาที ⏳)
ช่องนี้นานที่สุด **เปิดแท็บค้างไว้** อย่าปล่อยให้จอดับนานเกิน 90 นาที จดเวลา build ลงแบบฟอร์ม
ถ้าขึ้น `error` ให้ copy ข้อความ 30 บรรทัดสุดท้ายใส่แบบฟอร์มแล้วหยุด

In [ ]:
t0 = time.time()
!cd /content/koharu && set -o pipefail && cargo build --release -p koharu-pipeline --bin colab_batch 2>&1 | grep -E "^(error|warning: unused)|Finished" -A5 | tail -n 30
BIN = '/content/koharu/target/release/colab_batch'
assert os.path.exists(BIN), 'build ไม่สำเร็จ ดู error ด้านบน'
BUILD_MIN = (time.time() - t0) / 60
print(f'build เสร็จใน {BUILD_MIN:.1f} นาที')

## ขั้น 7 — อัปโหลดภาพตัวอย่าง 5–10 หน้า
กดปุ่ม Choose Files แล้วเลือกภาพ .jpg/.png/.webp จาก **มังงะปกติ** เท่านั้น
ควรเป็นหน้าที่ฟาร์มเคยแปลแล้ว จะได้เอาผลไปเทียบกันได้

In [ ]:
from google.colab import files
os.makedirs('/content/pages', exist_ok=True)
for name, data in files.upload().items():
    open(f'/content/pages/{name}', 'wb').write(data)
PAGES = sorted(p for p in os.listdir('/content/pages') if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
print(len(PAGES), 'หน้า:', PAGES)
assert 5 <= len(PAGES) <= 10, 'ใช้ 5–10 หน้า'

## ขั้น 8 — ตั้งค่าแบบเดียวกับฟาร์ม
ฟาร์มใช้ detect `koharu-layout-rfdetr-seg-2xl`, OCR `hayai-ocr`, inpaint `lama`, แปลด้วย Gemma 4 E4B Uncensored (ผ่าน LM Studio อีกเครื่อง)
บน Colab จะรัน LLM ในโปรเซสเดียวกันบน T4 แทน คำสั่งแปล (instructions) คัดมาจากฟาร์ม ตัด glossary เฉพาะเรื่องออก

หมายเหตุ: ฟาร์มตั้ง `repeat_penalty = 0.05` แต่ตัวเชื่อม LM Studio ไม่ได้ส่งค่านี้ไปจริง
ถ้าใส่ค่านี้กับ llama.cpp ในเครื่อง ค่าที่ต่ำกว่า 1 จะทำให้โมเดลพูดวนซ้ำ จึงปล่อยเป็นค่าเริ่มต้นของโมเดล (1.2)

In [ ]:
FARM_INSTRUCTIONS = """- Write numbers with Western digits (0-9), exactly as the source writes them. Do not convert them to the target language's own numeral system.

- Thai politeness particles: write none by default.
  Only add "khrap" or "kha" when something on this page tells you the speaker's gender. Do not infer it from the topic, the tone, or how polite the line sounds. A guess is wrong about half the time and a reader sees it instantly as the wrong character speaking.
  These are the only things that count as evidence:
    1. This segment's own words: the first-person pronouns boku, ore, jibun, washi, sessha, or the blunt endings da ze, da zo, da na, kai, kane, or a plain imperative. That speaker is male, so the particle is "khrap".
    2. This segment's own words: the first-person pronouns atashi, watakushi, or the endings kashira, da wa, desu wa, no yo, wa yo, wa ne, masu wa, kudasaimashi. That speaker is female, so the particle is "kha".
    3. Another segment on this page where the same character plainly speaks and shows one of the markers above.
    4. The series character sheet in the glossary below, when it names this speaker.
  When one of those four is present you MUST write the matching particle. Leaving it out there is as wrong as choosing the opposite one.
  When none of them is present, write polite Thai with no gendered particle at all. That is the correct answer, not a fallback.
  Never write the two particles together as a pair, with a slash or in any other form.
  A retainer or servant speaking in de gozaru or de gozaimasu is male, unless the character sheet says otherwise.
- Your output is Thai. Never copy a Japanese word, pronoun or name-suffix into it.

- Write an ellipsis once and only once: three dots. Japanese sets several ellipsis characters in a row for pacing, and copying that run into Thai reads as a stutter. Collapse any longer run to three dots. This rule is about dots only: leave "!!" and "!?" exactly as the source writes them, because Thai manga uses them the same way.

- The lines of one page reach you in reading order, and one sentence often runs across two or three of them. Measured on a real chapter, 65% of the Japanese lines do not finish their sentence. When a line does not finish, translate it so it joins the lines after it into one natural Thai sentence, broken at the same places. Do not pad a fragment into a whole sentence of its own, and never repeat in one line what you already wrote in the line before.

Write the translation of EVERY block in Thai script only. The output must contain no English words and no untranslated source text — if a block's source text is already in English, still rewrite it fully in Thai. Never leave any block in English or in its original language."""

import pandas as pd

def write_config(llm, quantization):
    quant_line = f'quantization = "{quantization}"' if quantization else ''
    os.makedirs(os.path.expanduser('~/.koharu'), exist_ok=True)
    open(os.path.expanduser('~/.koharu/config.toml'), 'w', encoding='utf-8').write(f"""
[pipeline.detection]
model = "koharu-layout-rfdetr-seg-2xl"

[pipeline.ocr]
model = "hayai-ocr"

[pipeline.inpainting]
model = "lama"

[pipeline.translation]
target_language = "th-TH"
instructions = '''{FARM_INSTRUCTIONS}'''

[pipeline.translation.generation]
temperature = 0.4
min_p = 0.1
reasoning = true
vision = true

[pipeline.translation.model]
provider = "local"
model = "{llm}"
{quant_line}
reasoning = true
vision = true
""")

GGUF = {}  # ไฟล์โมเดลที่โปรเซสเปิดจริงในแต่ละรอบ

def gguf_in_use(pid):
    """อ่านจาก /proc ว่าโปรเซสกำลังเปิดหรือ map ไฟล์ .gguf ไหนอยู่ = ไฟล์ที่โหลดจริง ไม่ใช่แค่ชื่อในตั้งค่า"""
    found = set()
    try:
        for line in open(f'/proc/{pid}/maps'):
            if line.rstrip().endswith('.gguf'):
                found.add(line.split(None, 5)[-1].strip())
        for fd in os.listdir(f'/proc/{pid}/fd'):
            try:
                target = os.readlink(f'/proc/{pid}/fd/{fd}')
            except OSError:
                continue
            if target.endswith('.gguf'):
                found.add(target)
    except OSError:
        pass
    return found

def run(llm, out, quantization=None):
    write_config(llm, quantization)
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    vram = subprocess.Popen(f'nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -lms 500 > {out}/vram.log', shell=True)
    t0 = time.time()
    seen = set()
    with open(f'{out}/log.txt', 'w') as log:
        p = subprocess.Popen([BIN, '--input', '/content/pages', '--output', out], stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            seen |= gguf_in_use(p.pid)
            time.sleep(0.5)
    wall = time.time() - t0
    vram.terminate()
    print(open(f'{out}/log.txt').read()[-4000:])
    GGUF[llm] = sorted(seen)
    print('ไฟล์ GGUF ที่โหลดจริง:')
    for f in GGUF[llm]:
        print(f'  {os.path.basename(f)}  ({os.path.getsize(f) / 2**30:.2f} GB)')
    if p.returncode != 0:
        raise RuntimeError('pipeline ล้มเหลว — copy ข้อความด้านบนลงแบบฟอร์ม')
    t = pd.read_csv(f'{out}/timings.tsv', sep='\t')
    t['sec'] = pd.to_numeric(t['load_s'], errors='coerce').fillna(0) + t['run_s']
    first = t['page'].iloc[0]
    warm = t[t['page'] != first]
    summary = pd.DataFrame({
        'หน้าแรก (เย็น) วินาที': t[t['page'] == first].groupby('stage', sort=False)['sec'].sum(),
        'หน้าถัดไป (อุ่น) เฉลี่ย วินาที': warm.groupby('stage', sort=False)['sec'].mean(),
    }).round(2)
    peak = max(int(x) for x in open(f'{out}/vram.log').read().split() if x.isdigit())
    per_page = summary.loc['total', 'หน้าถัดไป (อุ่น) เฉลี่ย วินาที']
    print(f'\n{llm}: ทั้งหมด {wall:.0f} วินาที | อุ่นแล้ว {per_page:.1f} วินาที/หน้า ≈ {3600 / per_page:.0f} หน้า/ชม. | VRAM สูงสุด {peak / 1024:.1f} GB')
    display(summary)
    return summary

## ขั้น 9 — รอบ A: โมเดลเดียวกับฟาร์ม (Gemma 4 E4B Uncensored)
ฟาร์มใช้ `Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf` (รุ่น uncensored ไม่ใช่ base)
รายการโมเดลของ koharu มีรุ่นเดียวกันแต่เป็น Q5_K_P (ไม่มี Q5_K_M) จึงตั้ง `Q5_K_P` ให้ใกล้ฟาร์มที่สุด
(รอบก่อนไม่ได้ตั้ง จึงได้ Q4_K_P ซึ่งเล็กกว่า) ท้ายช่องจะพิมพ์ชื่อไฟล์ GGUF ที่โหลดจริง

รอบแรกจะดาวน์โหลดโมเดลทั้งหมด (หลาย GB) เวลาของหน้าแรกจึงสูง ให้ดูค่า "อุ่น" เป็นหลัก

In [ ]:
LLM_A, QUANT_A = 'gemma4-e4b-uncensored', 'Q5_K_P'
RESULT_A = run(LLM_A, '/content/out_A', QUANT_A)

## ขั้น 10 — รอบ B: โมเดลใหญ่ขึ้นเพื่อดูคุณภาพ
ค่าเริ่มต้นคือ Gemma 4 12B Uncensored (ไฟล์ `Gemma4-12B-QAT-Uncensored-HauhauCS-Balanced-Q4_K_M.gguf`, ใส่ T4 ได้)
เปลี่ยนได้จากช่องเลือกด้านขวา ถ้าขึ้น out of memory ให้จดลงแบบฟอร์ม แล้วข้ามไปขั้น 11

In [ ]:
LLM_B = 'gemma4-12b-uncensored'  # @param ["gemma4-12b-uncensored", "gemma4-12b-it", "gemma4-e4b-it"]
QUANT_B = None  # ใช้ค่าเริ่มต้นของโมเดล
RESULT_B = run(LLM_B, '/content/out_B', QUANT_B)

## ขั้น 11 — ดูผลเทียบกัน
ภาพซ้าย = รอบ A, ขวา = รอบ B และตารางข้อความญี่ปุ่น → ไทย ของทั้งสองรอบ

In [ ]:
from IPython.display import HTML
import base64

def img(path):
    if not os.path.exists(path):
        return '<i>ไม่มี</i>'
    return f'<img src="data:image/png;base64,{base64.b64encode(open(path, "rb").read()).decode()}" style="width:100%">'

def show_results():
    for p in PAGES:
        stem = os.path.splitext(p)[0]
        display(HTML(f'<h4>{p}</h4><table><tr><th>A: {LLM_A}</th><th>B: {LLM_B}</th></tr>'
                     f'<tr><td style="width:50%">{img(f"/content/out_A/{stem}.png")}</td>'
                     f'<td style="width:50%">{img(f"/content/out_B/{stem}.png")}</td></tr></table>'))
    texts = pd.read_csv('/content/out_A/texts.tsv', sep='\t', quoting=3).rename(columns={'thai': 'thai_A'})
    if os.path.exists('/content/out_B/texts.tsv'):
        b = pd.read_csv('/content/out_B/texts.tsv', sep='\t', quoting=3)[['page', 'block', 'thai']].rename(columns={'thai': 'thai_B'})
        texts = texts.merge(b, on=['page', 'block'], how='left')
    pd.set_option('display.max_colwidth', None)
    display(texts)

show_results()

## ขั้น 12 — (ไม่บังคับ) render ใหม่อย่างเดียว ใช้คำแปลเดิม (ไม่ถึง 1 นาที)
วาดภาพใหม่จากงานที่เก็บไว้ในขั้น 9–10 **ไม่โหลดโมเดลและไม่แปลซ้ำ** ใช้เมื่อเปลี่ยนฟอนต์หรืออยากดูภาพใหม่
เขียน `<หน้า>.png`, `clean_<หน้า>.png` และ `blocks.json` ใหม่ทั้งหมด
ต้องเป็น runtime เดิมที่ยังไม่ได้ปิด (ถ้าปิดไปแล้ว งานที่เก็บไว้จะหายไปด้วย)

In [ ]:
for out in ['/content/out_A', '/content/out_B']:
    if not os.path.isdir(f'{out}/session.khrproj'):
        print(out, ': ไม่มีงานที่เก็บไว้ ข้าม')
        continue
    r = subprocess.run([BIN, '--render-only', '--output', out], text=True, capture_output=True)
    print(out, '\n', r.stderr[-2000:])
    if r.returncode != 0:
        raise RuntimeError('render ใหม่ล้มเหลว — copy ข้อความด้านบนส่งมา')
show_results()

## ขั้น 13 — ตรวจฟอนต์ไทยรอบสุดท้าย
วาดตัวอย่างภาษาไทยอีกครั้ง และดูว่า koharu เองรายงานว่าเจอฟอนต์ไทยในทุกรอบ ถ้าขึ้น ❌ ให้ดูภาพในขั้น 11 ว่ายังเป็นกล่องไหม

In [ ]:
THAI_FONT_OK = check_thai_font()
for out in ['/content/out_A', '/content/out_B']:
    if os.path.exists(f'{out}/log.txt'):
        line = next((l for l in open(f'{out}/log.txt') if l.startswith('Thai system fonts')), None)
        print(out, ':', line.strip() if line else '❌ koharu ไม่ได้รายงานฟอนต์ไทย')
        THAI_FONT_OK = THAI_FONT_OK and line is not None
print('✅ ผ่าน' if THAI_FONT_OK else '❌ ไม่ผ่าน')

## ขั้น 14 — ดาวน์โหลดผลแล้วปิดเครื่อง
ได้ไฟล์ `koharu_colab_results.zip` (ภาพที่แปลแล้ว, clean_*.png, blocks.json, timings.tsv, texts.tsv, vram.log, log.txt ของทั้งสองรอบ และ run_info.txt ที่มีชื่อไฟล์ GGUF ที่โหลดจริง)
จากนั้นกรอก `results-form.md` แล้วไปที่เมนู **Runtime → Disconnect and delete runtime** เพื่อคืนโควตา GPU

In [ ]:
gguf_lines = '\n'.join(f'gguf_{name}={os.path.basename(f)}' for name, paths in GGUF.items() for f in paths)
open('/content/run_info.txt', 'w').write(
    f'commit={COMMIT}\ncuda_fix={CUDA_FIX}\nbuild_min={BUILD_MIN:.1f}\npages={len(PAGES)}\n'
    f'llm_a={LLM_A}\nquant_a={QUANT_A}\nllm_b={LLM_B}\nquant_b={QUANT_B or "default"}\n{gguf_lines}\n'
    f'thai_font={sh("fc-match -f %{family} :lang=th").strip()}\nthai_font_ok={THAI_FONT_OK}\n{smi}')
print(open('/content/run_info.txt').read())
sh('cd /content && rm -f koharu_colab_results.zip && zip -qr koharu_colab_results.zip out_A out_B run_info.txt -x "*/session.khrproj/*"')
files.download('/content/koharu_colab_results.zip')